# Student Prep: Synthetic EEG Signal -> MNE Raw -> PSD

This short notebook prepares you for EEG spectral analysis.

You will do only four things:
1. Create a synthetic EEG-like signal.
2. Visualize it in time domain.
3. Convert it to an MNE `Raw` object.
4. Plot the power spectral density (PSD).

In [ ]:
! pip install mne #<--- run this

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne

The EEG can be imagined as a sum of multiple harmonic functions, such as sines and cosines, with different frequencies and amplitudes. Let's generate an artificial EEG signal.

## 1) Create a synthetic EEG-like signal

We combine a few oscillations and small random noise.

In [ ]:
sfreq = 250  # sampling frequency in Hz
duration = 20  # seconds
t = np.arange(0, duration, 1 / sfreq)

# Synthetic components of different frequencies
theta = 10 * np.sin(2 * np.pi * 5 * t)   # 5 Hz for alpha
alpha = 15 * np.sin(2 * np.pi * 10 * t)   # 10 Hz for alpha
beta = 10 * np.sin(2 * np.pi * 20 * t)    # 20 Hz for beta
gamma = 5 * np.sin(2 * np.pi * 45 * t)    # 45 Hz for gamma
drift = 3 * np.sin(2 * np.pi * 1 * t)     # 1 Hz slow drift
noise = np.random.randn(len(t))*10   # random noise

x = theta + alpha + beta + gamma + drift + noise
print(f'Signal shape: {x.shape}, duration: {duration}s, sfreq: {sfreq}Hz')

## 2) Visualize the signal in time domain

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(t, x, color='tab:blue', linewidth=1)
plt.xlim(0, 5)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (V)')
plt.title('Synthetic EEG-like signal (first 5 seconds)')
plt.tight_layout()
plt.show()

## 3) Create an `MNE Raw` object from the signal

[MNE-Python](https://mne.tools/) is an open-source Python library for working with brain-signal data such as **EEG, MEG, sEEG, ECoG, and NIRS**.

It is commonly used to:

- load neurophysiological recordings
- preprocess and clean signals
- visualize brain data
- perform analysis such as filtering, artifact removal, epoching, source estimation, statistics, and decoding

### Here, we will convert our synthetic EEG into an object that can be analysed using the MNE-Python tools.

MNE expects the signal array to have shape:

```python
(n_channels, n_times)

In [ ]:
print(x.shape)# "x" <-our EEG

In [ ]:
data = x[np.newaxis, :]  # shape: (1, n_times)

info = mne.create_info(
    ch_names=['O1'],
    sfreq=sfreq,
    ch_types=['eeg']
)

raw = mne.io.RawArray(data, info, verbose='ERROR')
raw

## 4) Plot PSD using MNE

**PSD (Power Spectral Density)** shows how much signal power is present at each frequency, helping us see which rhythms are strongest in the data.

So instead of looking at the signal over time, PSD helps you look at which frequencies are presented in the signal.
For EEG, PSD is useful because brain activity is often described in frequency bands like:

- delta
- theta
- alpha
- beta

In [ ]:
spectrum = raw.compute_psd(method='welch', fmin=1, fmax=45, n_fft=1024, )

# Quick PSD plot from MNE
spectrum.plot(average=True)
plt.show()

Look at the graph and hypothesise that these peaks show.